# 05 pi_0 权限 smoke 与训练门控

        这一节不急着启动长训练，而是先检查 PaliGemma、Hugging Face gated 权限、权重下载、数据加载和 1-step smoke。只有 smoke 通过后，正式训练才有意义。


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys


def find_topic_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "assets" / "metrics_snapshot.json").exists():
            return candidate
    raise RuntimeError("请从 AMD ROCm 专题目录或 notebooks 子目录启动 Jupyter。")


TOPIC_ROOT = find_topic_root()
ASSET_DIR = TOPIC_ROOT / "assets"
NOTEBOOK_DIR = TOPIC_ROOT / "notebooks"
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/path/to/every-embodied/mujoco_pnp"))
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/path/to/datasets/every_embodied"))
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", TOPIC_ROOT / "outputs"))
MODEL_ROOT = Path(os.environ.get("MODEL_ROOT", PROJECT_ROOT / "ckpt"))

print("TOPIC_ROOT =", TOPIC_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT =", DATA_ROOT)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("MODEL_ROOT =", MODEL_ROOT)


In [ ]:
try:
    from IPython.display import Image, Markdown, display
except Exception:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

    def Image(filename=None, width=None):
        return f"[image] {filename}"


def show_asset(filename, width=960):
    path = ASSET_DIR / filename
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f"缺少素材：{path}")


def md_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(str(x) for x in row) + " |")
    display(Markdown("\n".join(lines)))


## Checkpoint 1：权限检查命令模板


In [ ]:
print("""
cd "$PROJECT_ROOT"
HF_TOKEN_STDIN=1 REQUIRE_PROXY=1 ./install_hf_token_for_pi0.sh
""")


这个命令形态强调两件事：token 不写进 Notebook，脚本先验证 `whoami`、PaliGemma 和 `lerobot/pi0`，全部通过后再保存。


## Checkpoint 2：1-step smoke 证明什么


In [ ]:
rows = [
    ("gated model 权限", "能读取 PaliGemma / pi0 所需配置"),
    ("数据加载", "能读取 LeRobot 数据集和 observation/action key"),
    ("policy 构造", "pi0 policy 能初始化"),
    ("一次训练步", "forward/backward/optimizer step 能跑通"),
    ("checkpoint 写出", "输出目录出现 smoke checkpoint"),
]
md_table(["检查项", "通过后说明什么"], rows)


1-step smoke 不证明策略收敛，也不代表成功率。它只是正式训练前的门槛。


## Checkpoint 3：训练命令模板


In [ ]:
print("""
cd "$PROJECT_ROOT"
RUN_SMOKE=1 RUN_FULL_TRAIN=1 PI0_STEPS=20000 PI0_BATCH_SIZE=4 \\
  ./run_pi0_train_eval_after_hf_ready.sh
""")


正式训练建议放到终端、tmux 或后台脚本中执行。Notebook 负责记录参数、解释门控和读取训练后的 summary。


## Checkpoint 4：阶段性状态图


In [ ]:
show_asset("model_status_summary.png", width=900)


## Checkpoint 5：`tcp_to_plate` 后段 finisher 诊断


raw pi_0 目前还不能写成复刻成功。更有价值的一轮诊断是把后段 finisher 的 state 从 19 维扩到 22 维，额外加入 `tcp_to_plate = tcp_link_xyz - plate_xyz`。这一步让模型显式知道盘心相对 TCP 的方向，避免后段只靠图像和 phase 隐式猜目标。

        这个实验仍然是诊断 scaffold：前缀由 oracle 提供，schedule 从 `move_preplace` 对齐，finisher 使用 22D state。后续 30-seed 排查发现，短 `move_preplace` schedule 会提前 lower / release；强制使用长 schedule 模板后，改好版 scaffold 能稳定通过 30 个未见 seed。这个结果证明瓶颈被定位和修复，但不代表 raw pi_0 已经端到端成功。

        下一步开始拆 scaffold。第一个被替换的是手写 gripper 规则：用 phase-only logistic head 预测夹爪开闭，EEF/arm 仍由 22D finisher 输出。full-state gripper head 虽然训练集准确率 `100%`，但闭环上线失败；phase-only head 只看 `timestamp + phase_index_norm + phase_onehot11`，在完整 unseen seeds `1010-1039` 上仍保持 strict `30/30`。这个对照说明，小数据短事件 head 的输入要克制，不要把闭环会漂移的机械臂状态全塞进去。

        第二块脚手架先用固定阈值 adaptive gate 替代强制长 `move_preplace` 模板，完整 unseen seeds `1010-1039` 仍是 strict `30/30`，但其中一部分依赖 `max_steps=180` 安全兜底。继续往前走，我们训练了 logistic transition head。第一版 full head 训练集准确率约 `99.27%`，但 seed `1010` 闭环早放失败；第二版只保留 `tcp_to_plate_xy + local_step_norm`，训练集准确率约 `95.01%`，却在完整 unseen seeds `1010-1039` 上达到 strict `30/30`、legacy `30/30`。30 条里 `29/30` 是 transition head 主动触发，只有 seed `1021` 走到 max-step 安全兜底。这个对照说明，小数据 phase head 的输入也要克制，稳定几何线索比训练集高准确率更重要。

        第三层继续拆前段 contact primitive。naive all-head 把所有 transition 都按 phase tail 学，训练集指标不差，但 smoke `1010/1011` 变成 `0/2`，因为 `descend_to_close` 会在 TCP 还高出抓取 floor 约 `0.08 m` 时提前触发。修复版把 `pregrasp_to_descend` 改成几何标签，特征加入 TCP 到 grasp / pregrasp / floor 的相对量，并在部署时加 `descend_floor_guard`。完整 unseen seeds `1010-1039` 为 strict `30/30`、legacy `30/30`；核心切换 `pregrasp->descend`、`descend->close`、`close->lift` 都是 `30/30` 由 head 触发，floor guard 共阻挡 `342` 次高空 close。这仍是 contact scaffold，不是 raw pi_0 端到端成功。

        第四层拆掉 `dataset_schedule` 尾段，改成 `dynamic_timed` finisher。第一次完整 30 seed 只有 strict `27/30`，失败 seeds `1021/1031/1036` 看起来像固定 dwell 不合适；复盘代码后发现根因是 `--tcpplate-prefix-target-state` 这个全局开关越界了：prefix 需要 `tcp_to_target`，但 dynamic finisher 也误用了它，而 finisher 训练时应该吃 `tcp_to_plate`。把 state builder 改成 stage-aware 后，prefix 阶段用 target-relative，finisher 阶段用 plate-relative；三个失败 seed 复测 `3/3`，旧版同一环境连续跑完整 unseen seeds `1010-1039` 回到 strict `30/30`、legacy `30/30`。但继续 trace 又发现，连续评估时前一个 episode 的 MuJoCo qvel / ctrl / free-joint 动态残留可能污染下一个 seed，seed `1035` 曾出现采样范围外的初始位置。于是 evaluator 增加 `--fresh-env-per-episode` 和 `--hard-reset-sim-data`。最终用 `--hard-reset-sim-data` clean protocol 重跑完整 unseen seeds `1010-1039`，仍是 strict `30/30`、legacy `30/30`，红杯 `18/18`、蓝杯 `12/12`；mean `xy=0.0219 m`，max `xy=0.0450 m`，最小 lift `0.1093 m`，最小 upright cos `0.9504`。之前 seed `1036` 的 `0.0993 m` 更像 reset 残留造成的评估伪影，不再作为边界样本处理。


In [ ]:
rows = [
    ("raw full20 open-loop", "1/20 final，3/20 ever", "raw policy 仍未过关"),
    ("raw closed-loop strict", "0/20", "最接近部署口径"),
    ("template-tail full20", "4/20", "固定尾段只能救部分样本"),
    ("旧 19D finisher handoff", "prefix 120/180/240/300 均为 0/2", "后段目标仍偏在杯子侧"),
    ("22D tcp_to_plate finisher", "strict 5/10，legacy 6/10", "plate-relative state 是有效线索，但仍是 scaffold"),
    ("22D + phase-scripted gripper", "strict 7/10，legacy 7/10", "夹爪时序能救一批失败，剩余主要是红杯落点"),
    ("unseen 30，schedule 0..9 重复", "strict 21/30，legacy 23/30", "失败全部来自短 move_preplace 模板"),
    ("unseen 30，强制长 schedule episode 0", "strict 30/30，legacy 30/30", "改好版 scaffold 稳定，但仍不是 raw pi_0"),
    ("长 schedule + phase-only learned gripper head", "strict 30/30，legacy 30/30", "第一块脚手架被可学习 head 替代"),
    ("schedule 0..9 + adaptive move gate + learned gripper", "strict 30/30，legacy 30/30", "第二块脚手架工程版：不再强制长 episode0"),
    ("schedule 0..9 + xy-step transition head + learned gripper", "strict 30/30，legacy 30/30", "第二块脚手架可学习版：29/30 主动触发"),
    ("policy prefix + contact transition heads + floor guard", "strict 30/30，legacy 30/30", "第三层去脚手架：核心接触切换由 head 触发，但仍保留 scaffold"),
    ("stage-aware dynamic_timed finisher + hard-reset clean protocol", "strict 30/30，legacy 30/30", "第四层去脚手架：prefix 用 target，finisher 用 plate；mean xy=0.0219 m，max xy=0.0450 m"),
]
md_table(["口径", "结果", "怎么理解"], rows)


In [ ]:
print("""
cd "$PROJECT_ROOT"

# 1. 把 19D phase-state 数据转换成 22D tcp_to_plate 数据。
python code/pi0/build_lerobot_state_phase_tcpplate.py \\
  --src-repo-id datawhale_eai_pnp_pi0_dagger_prefix120_step1500_state_phase_eef_abs_10eps_v1 \\
  --src-root ./demo_data_pi0_dagger_prefix120_step1500_state_phase_eef_abs_10eps_v1 \\
  --dst-repo-id datawhale_eai_pnp_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --dst-root ./demo_data_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --overwrite

# 2. 评估 22D finisher。注意：这个命令需要已有 22D checkpoint。
PYTHONPATH="$PWD:$TOPIC_ROOT/code/pi0:${PYTHONPATH:-}" \\
python code/pi0/evaluate_pi0_two_stage_tcpplate.py \\
  --tcpplate-base-evaluator code/pi0/evaluate_pi0_two_stage_eef_abs.py \\
  --tcpplate-schedule-start-phase move_preplace \\
  --prefix-source oracle \\
  --prefix-steps 180 \\
  --finisher-phase-state dataset_schedule \\
  --finisher-policy-path "$MODEL_ROOT/pi0_tcpplate_finisher/checkpoints/000250/pretrained_model" \\
  --finisher-dataset-repo-id datawhale_eai_pnp_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --finisher-dataset-root ./demo_data_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --seeds 1000,1001,1002,1003,1004,1005,1006,1007,1008,1009 \\
  --finisher-phase-schedule-episodes 0,1,2,3,4,5,6,7,8,9 \\
  --output-jsonl "$OUTPUT_ROOT/pi0_tcpplate_eval/results.jsonl" \\
  --summary-json "$OUTPUT_ROOT/pi0_tcpplate_eval/summary.json"

# 3. 更大样本量复核时，可以强制使用长 move_preplace 模板。
PYTHONPATH="$PWD:$TOPIC_ROOT/code/pi0:${PYTHONPATH:-}" \\
python code/pi0/evaluate_pi0_two_stage_tcpplate.py \\
  --tcpplate-base-evaluator code/pi0/evaluate_pi0_two_stage_eef_abs.py \\
  --tcpplate-schedule-start-phase move_preplace \\
  --tcpplate-force-schedule-episode 0 \\
  --prefix-source oracle \\
  --prefix-steps 180 \\
  --finisher-phase-state dataset_schedule \\
  --finisher-scripted-gripper \\
  --finisher-policy-path "$MODEL_ROOT/pi0_tcpplate_finisher/checkpoints/000250/pretrained_model" \\
  --finisher-dataset-repo-id datawhale_eai_pnp_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --finisher-dataset-root ./demo_data_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --seeds 1010,1011,1012,1013,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023,1024,1025,1026,1027,1028,1029,1030,1031,1032,1033,1034,1035,1036,1037,1038,1039 \\
  --output-jsonl "$OUTPUT_ROOT/pi0_tcpplate_eval_unseen30_long_schedule/results.jsonl" \\
  --summary-json "$OUTPUT_ROOT/pi0_tcpplate_eval_unseen30_long_schedule/summary.json"

# 4. 训练 phase-only gripper head。这个 head 只替代 gripper 规则，不改 EEF/arm。
python code/pi0/train_tcpplate_gripper_head.py \\
  --dataset-repo-id datawhale_eai_pnp_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --dataset-root ./demo_data_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --feature-mode phase \\
  --label-source action \\
  --output "$MODEL_ROOT/pi0_tcpplate_finisher/gripper_head_phase_logreg.npz" \\
  --summary-json "$MODEL_ROOT/pi0_tcpplate_finisher/gripper_head_phase_logreg_summary.json"

# 5. 用 learned gripper head 复核 30 个 unseen seed。
PYTHONPATH="$PWD:$TOPIC_ROOT/code/pi0:${PYTHONPATH:-}" \\
python code/pi0/evaluate_pi0_two_stage_tcpplate.py \\
  --tcpplate-base-evaluator code/pi0/evaluate_pi0_two_stage_eef_abs.py \\
  --tcpplate-schedule-start-phase move_preplace \\
  --tcpplate-force-schedule-episode 0 \\
  --tcpplate-gripper-head-path "$MODEL_ROOT/pi0_tcpplate_finisher/gripper_head_phase_logreg.npz" \\
  --prefix-source oracle \\
  --prefix-steps 180 \\
  --finisher-phase-state dataset_schedule \\
  --finisher-scripted-gripper \\
  --finisher-policy-path "$MODEL_ROOT/pi0_tcpplate_finisher/checkpoints/000250/pretrained_model" \\
  --finisher-dataset-repo-id datawhale_eai_pnp_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --finisher-dataset-root ./demo_data_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --seeds 1010,1011,1012,1013,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023,1024,1025,1026,1027,1028,1029,1030,1031,1032,1033,1034,1035,1036,1037,1038,1039 \\
  --output-jsonl "$OUTPUT_ROOT/pi0_tcpplate_eval_unseen30_gripper_head/results.jsonl" \\
  --summary-json "$OUTPUT_ROOT/pi0_tcpplate_eval_unseen30_gripper_head/summary.json"

# 6. 去掉强制长 episode0，改用 adaptive move_preplace gate。
# 这个版本使用 schedule 0..9 正常重复；如果 live TCP-to-plate 还太远，
# 就 hold move_preplace，直到 xy 足够小或达到保守 max-step 兜底。
PYTHONPATH="$PWD:$TOPIC_ROOT/code/pi0:${PYTHONPATH:-}" \\
python code/pi0/evaluate_pi0_two_stage_tcpplate.py \\
  --tcpplate-base-evaluator code/pi0/evaluate_pi0_two_stage_eef_abs.py \\
  --tcpplate-schedule-start-phase move_preplace \\
  --tcpplate-adaptive-move-preplace \\
  --tcpplate-adaptive-move-preplace-xy-threshold 0.05 \\
  --tcpplate-adaptive-move-preplace-min-steps 20 \\
  --tcpplate-adaptive-move-preplace-max-steps 180 \\
  --tcpplate-gripper-head-path "$MODEL_ROOT/pi0_tcpplate_finisher/gripper_head_phase_logreg.npz" \\
  --prefix-source oracle \\
  --prefix-steps 180 \\
  --finisher-phase-state dataset_schedule \\
  --finisher-scripted-gripper \\
  --finisher-policy-path "$MODEL_ROOT/pi0_tcpplate_finisher/checkpoints/000250/pretrained_model" \\
  --finisher-dataset-repo-id datawhale_eai_pnp_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --finisher-dataset-root ./demo_data_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --seeds 1010,1011,1012,1013,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023,1024,1025,1026,1027,1028,1029,1030,1031,1032,1033,1034,1035,1036,1037,1038,1039 \\
  --finisher-phase-schedule-episodes 0,1,2,3,4,5,6,7,8,9,0,1,2,3,4,5,6,7,8,9,0,1,2,3,4,5,6,7,8,9 \\
  --output-jsonl "$OUTPUT_ROOT/pi0_tcpplate_eval_unseen30_adaptive_gate/results.jsonl" \\
  --summary-json "$OUTPUT_ROOT/pi0_tcpplate_eval_unseen30_adaptive_gate/summary.json"

# 7. 训练 xy-step transition head。full-state transition head 训练集更准，
# 但会被高度/方向相关性误导；稳定版只用 tcp_to_plate_xy + local_step_norm。
python code/pi0/train_tcpplate_transition_head.py \\
  --dataset-repo-id datawhale_eai_pnp_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --dataset-root ./demo_data_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --feature-mode xy_step \\
  --step-scale 180 \\
  --output "$MODEL_ROOT/pi0_tcpplate_finisher/transition_head_tcpplate_xy_step_logreg.npz" \\
  --summary-json "$MODEL_ROOT/pi0_tcpplate_finisher/transition_head_tcpplate_xy_step_logreg_summary.json"

# 8. 用 learned transition head 替代固定阈值 gate，schedule 0..9 正常重复。
PYTHONPATH="$PWD:$TOPIC_ROOT/code/pi0:${PYTHONPATH:-}" \\
python code/pi0/evaluate_pi0_two_stage_tcpplate.py \\
  --tcpplate-base-evaluator code/pi0/evaluate_pi0_two_stage_eef_abs.py \\
  --tcpplate-schedule-start-phase move_preplace \\
  --tcpplate-transition-head-path "$MODEL_ROOT/pi0_tcpplate_finisher/transition_head_tcpplate_xy_step_logreg.npz" \\
  --tcpplate-transition-head-threshold 0.5 \\
  --tcpplate-adaptive-move-preplace-min-steps 20 \\
  --tcpplate-adaptive-move-preplace-max-steps 180 \\
  --tcpplate-gripper-head-path "$MODEL_ROOT/pi0_tcpplate_finisher/gripper_head_phase_logreg.npz" \\
  --prefix-source oracle \\
  --prefix-steps 180 \\
  --finisher-phase-state dataset_schedule \\
  --finisher-scripted-gripper \\
  --finisher-policy-path "$MODEL_ROOT/pi0_tcpplate_finisher/checkpoints/000250/pretrained_model" \\
  --finisher-dataset-repo-id datawhale_eai_pnp_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --finisher-dataset-root ./demo_data_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --seeds 1010,1011,1012,1013,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023,1024,1025,1026,1027,1028,1029,1030,1031,1032,1033,1034,1035,1036,1037,1038,1039 \\
  --finisher-phase-schedule-episodes 0,1,2,3,4,5,6,7,8,9,0,1,2,3,4,5,6,7,8,9,0,1,2,3,4,5,6,7,8,9 \\
  --output-jsonl "$OUTPUT_ROOT/pi0_tcpplate_eval_unseen30_transition_head/results.jsonl" \\
  --summary-json "$OUTPUT_ROOT/pi0_tcpplate_eval_unseen30_transition_head/summary.json"

# 9. 训练 target-relative contact transition heads。
# naive all-head 会学到“阶段快结束了”，不等于“现在可以安全闭爪”；
# 这里用 pregrasp geometry label + grasp/floor geometry features。
python code/pi0/train_tcptarget_contact_transition_head.py \\
  --dataset-repo-id datawhale_eai_pnp_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --dataset-root ./demo_data_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --task-set all \\
  --early-label-mode pregrasp_geometry \\
  --feature-mode grasp_geom_count_step \\
  --step-scale 180 \\
  --output-npz "$MODEL_ROOT/pi0_tcpplate_finisher/contact_transition_head_pregraspgeom_graspgeom_step.npz" \\
  --summary-json "$MODEL_ROOT/pi0_tcpplate_finisher/contact_transition_head_pregraspgeom_graspgeom_step_summary.json"

# 10. 用 contact transition heads 替代前段 pregrasp/descend/close 的部分手写切换。
# descend_floor_guard 仍保留，防止 TCP 还明显高于抓取 floor 时提前闭爪。
PYTHONPATH="$PWD:$TOPIC_ROOT/code/pi0:${PYTHONPATH:-}" \\
python code/pi0/evaluate_pi0_two_stage_tcpplate.py \\
  --tcpplate-base-evaluator code/pi0/evaluate_pi0_two_stage_eef_abs.py \\
  --tcpplate-schedule-start-phase move_preplace \\
  --tcpplate-prefix-target-state \\
  --tcpplate-contact-primitive guided_contact_lift \\
  --tcpplate-contact-transition-head-path "$MODEL_ROOT/pi0_tcpplate_finisher/contact_transition_head_pregraspgeom_graspgeom_step.npz" \\
  --tcpplate-contact-transition-head-threshold 0.5 \\
  --tcpplate-contact-pregrasp-hold 999 \\
  --tcpplate-contact-descend-hold 999 \\
  --tcpplate-contact-transition-head-strict \\
  --tcpplate-contact-descend-floor-guard \\
  --tcpplate-gripper-head-path "$MODEL_ROOT/pi0_tcpplate_finisher/gripper_head_phase_logreg.npz" \\
  --tcpplate-transition-head-path "$MODEL_ROOT/pi0_tcpplate_finisher/transition_head_tcpplate_xy_step_logreg.npz" \\
  --tcpplate-adaptive-move-preplace-min-steps 20 \\
  --tcpplate-adaptive-move-preplace-max-steps 180 \\
  --prefix-source policy \\
  --prefix-policy-path "$MODEL_ROOT/pi0_prefix_tcptarget/checkpoints/001000/pretrained_model" \\
  --prefix-dataset-repo-id datawhale_eai_pnp_fullprefix_oracle_prefix_until_mpreplace_state_phase_tcptarget_eef_abs_10eps_v1 \\
  --prefix-dataset-root ./demo_data_pi0_fullprefix_oracle_prefix_until_mpreplace_state_phase_tcptarget_eef_abs_10eps_v1 \\
  --prefix-steps 180 \\
  --finisher-phase-state dataset_schedule \\
  --finisher-scripted-gripper \\
  --finisher-policy-path "$MODEL_ROOT/pi0_tcpplate_finisher/checkpoints/000250/pretrained_model" \\
  --finisher-dataset-repo-id datawhale_eai_pnp_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --finisher-dataset-root ./demo_data_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --seeds 1010,1011,1012,1013,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023,1024,1025,1026,1027,1028,1029,1030,1031,1032,1033,1034,1035,1036,1037,1038,1039 \\
  --finisher-phase-schedule-episodes 0,1,2,3,4,5,6,7,8,9,0,1,2,3,4,5,6,7,8,9,0,1,2,3,4,5,6,7,8,9 \\
  --output-jsonl "$OUTPUT_ROOT/pi0_contact_transition_head_unseen30/results.jsonl" \\
  --summary-json "$OUTPUT_ROOT/pi0_contact_transition_head_unseen30/summary.json"

# 11. 拆掉 dataset_schedule 尾段，改用 stage-aware dynamic_timed finisher。
# 关键点：--tcpplate-prefix-target-state 只让 prefix 使用 tcp_to_target；
# stage-aware evaluator 会让 finisher 回到 tcp_to_plate。
PYTHONPATH="$PWD:$TOPIC_ROOT/code/pi0:${PYTHONPATH:-}" \\
python code/pi0/evaluate_pi0_two_stage_tcpplate.py \\
  --tcpplate-base-evaluator code/pi0/evaluate_pi0_two_stage_eef_abs.py \\
  --tcpplate-prefix-target-state \\
  --tcpplate-contact-primitive guided_contact_lift \\
  --tcpplate-contact-transition-head-path "$MODEL_ROOT/pi0_tcpplate_finisher/contact_transition_head_pregraspgeom_graspgeom_step.npz" \\
  --tcpplate-contact-transition-head-threshold 0.5 \\
  --tcpplate-contact-pregrasp-hold 999 \\
  --tcpplate-contact-descend-hold 999 \\
  --tcpplate-contact-transition-head-strict \\
  --tcpplate-contact-descend-floor-guard \\
  --tcpplate-gripper-head-path "$MODEL_ROOT/pi0_tcpplate_finisher/gripper_head_phase_logreg.npz" \\
  --prefix-source policy \\
  --prefix-policy-path "$MODEL_ROOT/pi0_prefix_tcptarget/checkpoints/001000/pretrained_model" \\
  --prefix-dataset-repo-id datawhale_eai_pnp_fullprefix_oracle_prefix_until_mpreplace_state_phase_tcptarget_eef_abs_10eps_v1 \\
  --prefix-dataset-root ./demo_data_pi0_fullprefix_oracle_prefix_until_mpreplace_state_phase_tcptarget_eef_abs_10eps_v1 \\
  --prefix-steps 180 \\
  --finisher-phase-state dynamic_timed \\
  --finisher-start-phase move_preplace \\
  --timed-phase-dwell-spec move_preplace:260,lower_to_plate:40,retreat:40 \\
  --hard-reset-sim-data \\
  --finisher-scripted-gripper \\
  --finisher-policy-path "$MODEL_ROOT/pi0_tcpplate_finisher/checkpoints/000250/pretrained_model" \\
  --finisher-dataset-repo-id datawhale_eai_pnp_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --finisher-dataset-root ./demo_data_pi0_dagger_prefix120_step1500_state_phase_tcpplate_eef_abs_10eps_v1 \\
  --seeds 1010,1011,1012,1013,1014,1015,1016,1017,1018,1019,1020,1021,1022,1023,1024,1025,1026,1027,1028,1029,1030,1031,1032,1033,1034,1035,1036,1037,1038,1039 \\
  --output-jsonl "$OUTPUT_ROOT/pi0_dynamic_timed_stageaware_unseen30/results.jsonl" \\
  --summary-json "$OUTPUT_ROOT/pi0_dynamic_timed_stageaware_unseen30/summary.json"
""")


如果要跑夹爪脚本化对照，在评估命令里额外加入 `--finisher-scripted-gripper`。这一项只改 gripper，不改 EEF/arm，用来判断开闭爪时序是不是主要瓶颈。`--tcpplate-force-schedule-episode 0` 用于复核长 `move_preplace` 模板是否能消掉提前释放问题。`--tcpplate-gripper-head-path` 会在 `--finisher-scripted-gripper` 的钩子里加载 learned head，把手写 phase 规则替换成可训练的小模型。`--tcpplate-adaptive-move-preplace` 把强制长模板换成 live progress gate；`--tcpplate-transition-head-path` 则把固定阈值 gate 换成 logistic transition head；`--tcpplate-contact-transition-head-path` 进一步替代前段 contact primitive 的部分 phase 切换。`dynamic_timed` 版本要特别注意 stage-aware state：prefix 可以用 `tcp_to_target`，finisher 必须用 `tcp_to_plate`。批量评估建议加 `--hard-reset-sim-data`，先清掉跨 episode 的 MuJoCo 动力学残留；`--fresh-env-per-episode` 更干净但反复创建环境时可能遇到资产 provider 偶发报错。当前 stage-aware dynamic finisher 在 hard-reset clean protocol 的 30 个 unseen seed 上是 strict `30/30`，但仍保留 target-relative contact scaffold、floor guard、learned gripper head 和后段 finisher。


如果旧 checkpoint 的 state normalizer 是 19 维，而新数据是 22 维，会出现 `mean/std` shape mismatch。诊断实验里可以复制 checkpoint 并删除 `normalize_inputs.buffer_observation_state.mean/std`，让 22D 数据集统计重新初始化；不要把这个临时处理误写成模型结构创新。
